# M3 · Ejecución integral del sistema RAG clínico

Une el trabajo de **todo el equipo** para el Módulo 3 y corre el pipeline de
punta a punta:

```
corpus (Isa)  →  retrieval + orquestación (Pau)  →  generación (Agustín)  →  RAGAS + cruce con M2 (Luis)
```

Sigue el patrón de `M2/ejecucion/start_inference_harness.ipynb`: clona el repo,
instala dependencias, monta Drive, genera el `.env` y llama a los entry points
(`run_corpus.py`, `experimento_s08.py`, `run_generacion.py`, `run_ragas.py`).

**Antes de correr:**
1. Activar GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.
2. Tener en Secretos de Colab (icono de la llave) `GROQ_API_KEY` y, si se quiere
   la tool de normalización real, `BIOPORTAL_API_KEY`.
3. El corpus y el índice viven en Drive (`DATA_ROOT`); el código se clona aparte.

## 1 · Parámetros de la corrida

Todo lo configurable del notebook está aquí. Las rutas de cada pieza viven en
**su propio YAML** dentro del repo (no se duplican aquí).

In [ ]:
RAMA        = "dev"
REPO_URL    = "https://github.com/luisNP21/Topicos-IA.git"
REPO_DIR    = "/content/Topicos-IA"

# Datos (corpus, índice, salidas) en Drive; el código vive en REPO_DIR.
DATA_ROOT   = "/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3"

# Eval set compartido (15 casos: entidad + pregunta + esperado). Si no existe,
# se usa el de ejemplo de la generación para no frenar la corrida.
EJEMPLOS    = "M3/ejecucion/eval_set_casos.json"
EJEMPLOS_ALT = "M3/generacion/entidades_ejemplo.json"

MODO_RAGAS         = "real"   # "real" (librería ragas + juez Groq) | "mock"
CORRER_CORPUS      = True     # False si el índice ya está construido en Drive
CORRER_EXPERIMENTO = True     # calibra umbrales y mide el delta del retrieval

print("Rama:", RAMA, "| repo:", REPO_DIR, "| datos:", DATA_ROOT,
      "| RAGAS:", MODO_RAGAS, "| corpus:", CORRER_CORPUS, "| experimento:", CORRER_EXPERIMENTO)


## 2 · Clonar el repositorio

Se clona (o se sincroniza) la rama `dev`, que integra el trabajo de los cuatro.

In [ ]:
import os, subprocess

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} falló:\n{r.stderr.strip()}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    git("fetch", "origin", RAMA)
    git("checkout", "-B", RAMA, f"origin/{RAMA}")
    git("reset", "--hard", f"origin/{RAMA}")
else:
    subprocess.run(["git", "clone", "--branch", RAMA, REPO_URL, REPO_DIR], check=True)

%cd {REPO_DIR}
print(git("log", "-1", "--oneline"))

## 3 · Instalar dependencias

Cada pieza trae su `requirements.txt`. El retrieval no trae uno propio, así que
sus dependencias se instalan aparte.

In [ ]:
%pip install -q -r M3/corpus/requirements.txt
%pip install -q -r M3/tools/requirements.txt
%pip install -q -r M3/generacion/requirements.txt
%pip install -q -r M3/ragas/requirements.txt
# Dependencias del retrieval (no tiene requirements.txt propio):
%pip install -q sentence-transformers chromadb rank_bm25 groq python-dotenv PyYAML
print("Dependencias instaladas.")

## 4 · Montar Drive y verificar recursos

Comprueba que estén los insumos: las guías (PDF), el resultado de M2 para el
cruce y el eval set de M3.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA   = Path(DATA_ROOT)
REPO   = Path(REPO_DIR)
ejemplos_repo = REPO / EJEMPLOS
ejemplos_alt  = REPO / EJEMPLOS_ALT

checks = {
    "Carpeta de datos en Drive": DATA,
    "Guías PDF (data/guias_clinicas)": DATA / "data/guias_clinicas",
    "Resultado M2 (para el cruce)": REPO / "M2/ejecucion/outputs_gold/resultado_dimension1.json",
    "Eval set del equipo": ejemplos_repo,
}
for nombre, p in checks.items():
    print(f"{nombre}: {'OK' if p.exists() else 'FALTA'} -- {p}")

if ejemplos_repo.exists():
    ENTIDADES = EJEMPLOS
elif ejemplos_alt.exists():
    ENTIDADES = EJEMPLOS_ALT
    print(f"AVISO: no existe {EJEMPLOS}; se usará {EJEMPLOS_ALT} (ejemplos de prueba).")
else:
    raise FileNotFoundError("No hay ningún eval set de entidades para la generación.")

## 5 · Secretos y `.env`

`PROJECT_ROOT` apunta al repo (generación y RAGAS resuelven rutas desde ahí).
El corpus se corre con `PROJECT_ROOT = DATA_ROOT`, porque sus datos viven en Drive.

In [ ]:
import os
from google.colab import userdata

lineas = [f"PROJECT_ROOT={REPO_DIR}"]
for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
    try:
        valor = userdata.get(var)
        os.environ[var] = valor
        lineas.append(f"{var}={valor}")
        print(f"{var}: configurada")
    except Exception:
        print(f"{var}: no encontrada en Secretos" +
              (" (el juez y la generación fallarán en modo real)" if var == "GROQ_API_KEY"
               else " (la tool de normalización correrá en modo mock)"))

with open(os.path.join(REPO_DIR, ".env"), "w") as f:
    f.write("\n".join(lineas) + "\n")
print(".env generado (solo en este runtime; el archivo está ignorado por git)")

## 6 · Pruebas unitarias

Verifican la lógica de retrieval y de generación con modelos simulados, sin
descargar nada.

In [ ]:
!cd M3/retrieval && python -m unittest test_retrieval
!cd M3/generacion && python -m unittest test_generacion

## 7 · Corpus (Isa): ingesta de las guías + índice

Ingesta → chunking → embeddings → índice Chroma + manifest. Es **idempotente**:
las fuentes cuyo `doc_id` ya tiene chunks en Drive se saltan, así que se puede
volver a correr sin reindexar todo.

In [ ]:
if CORRER_CORPUS:
    import shutil
    # Las guías vienen en el repo; la ingesta las lee desde Drive (PROJECT_ROOT/data).
    destino = DATA / "data/guias_clinicas"
    destino.mkdir(parents=True, exist_ok=True)
    origen = REPO / "M3/corpus/guias_clinicas"
    for pdf in sorted(origen.glob("*.pdf")):
        if not (destino / pdf.name).exists():
            shutil.copy2(pdf, destino / pdf.name)
            print("copiada:", pdf.name)
    %cd {REPO_DIR}/M3/corpus
    !PROJECT_ROOT={DATA_ROOT} python run_corpus.py --config config.yaml --fuentes fuentes.yaml
    %cd {REPO_DIR}
else:
    print("CORRER_CORPUS=False: se usa el índice que ya está en Drive.")

## 8 · Retrieval (Pau): calibración de umbrales y delta

Corre el experimento controlado (RAG ingenuo vs. híbrido vs. reranker vs. sistema
completo con normalización). Escribe en `M3/retrieval/outputs/`:

- `deltas_s08.csv` — el **delta medido** contra el RAG ingenuo (rúbrica).
- `umbrales.json` — umbrales calibrados que consume la orquestación.
- `delta_naive_vs_advanced.json` — contrato de salida para el reporte.

In [ ]:
if CORRER_EXPERIMENTO:
    import subprocess, sys
    # El experimento escribe umbrales.json y deltas_s08.csv en M3/retrieval/outputs.
    subprocess.run([sys.executable, "experimento_s08.py", "--corpus", "real"],
                   cwd=f"{REPO_DIR}/M3/retrieval", check=True)
else:
    print("CORRER_EXPERIMENTO=False: se usan los umbrales provisionales del YAML.")


## 9 · Generación (Agustín): respuesta final del RAG

Encadena orquestación (`resolver_query`: cuándo se invoca la tool y qué hace si
falla) + generación (`generar_respuesta`). Consume el retrieval de Pau y la tool
de Luis desde sus YAML. Escribe `M3/outputs/resultado_generacion.json`, que es
justo lo que consume RAGAS.

In [ ]:
%cd {REPO_DIR}/M3/generacion
!PROJECT_ROOT={REPO_DIR} python run_generacion.py --config config.yaml --entidades {ENTIDADES}
%cd {REPO_DIR}

## 10 · RAGAS + cruce con el harness de M2 (Luis)

Calcula **faithfulness, context precision, context recall y answer relevancy** y
los cruza con el F1 de extracción de M2 para diagnosticar dónde falla el sistema
(extracción, corpus/retrieval o generación).

In [ ]:
%cd {REPO_DIR}/M3/ragas
!PROJECT_ROOT={REPO_DIR} python run_ragas.py --config config.yaml --modo {MODO_RAGAS}
%cd {REPO_DIR}

## 11 · Persistir los artefactos

El repo clonado se pierde al cerrar la sesión. Esta celda copia los resultados a
Drive y arma un zip descargable.

In [ ]:
import shutil
from pathlib import Path

persist = DATA / "resultados_pipeline"
persist.mkdir(parents=True, exist_ok=True)

for origen in [REPO / "M3/outputs", REPO / "M3/retrieval/outputs"]:
    if origen.exists():
        destino = persist / origen.name if origen.name != "outputs" else persist / "retrieval_outputs"
        shutil.copytree(origen, destino, dirs_exist_ok=True)
        print("copiado a Drive:", destino)

zip_path = shutil.make_archive("/content/resultados_M3", "zip", persist)
print("zip listo:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Descarga manual disponible en:", zip_path, "|", e)